In [ ]:
!pip install catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 27.8 MB/s eta 0:00:00


In [ ]:
import kagglehub
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score
from sklearn.utils import resample
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import warnings
from tqdm import tqdm
warnings.filterwarnings('ignore')

In [ ]:
def evaluate_with_bootstrap(model, X_test, y_test, n_iterations=1000):
    metrics = {'roc_auc': [], 'f1': [], 'accuracy': [], 'precision': [], 'recall': []}

    is_multiclass = len(np.unique(y_test)) > 2
    avg_method = 'macro' if is_multiclass else 'binary'

    for _ in range(n_iterations):
        # Случайная выборка с возвращением
        X_boot, y_boot = resample(X_test, y_test)

        # Пропуск, если в выборке только один класс (для предотвращения ошибки вычисления ROC)
        if len(np.unique(y_boot)) < 2: continue

        y_pred = model.predict(X_boot)
        y_prob = model.predict_proba(X_boot)

        try:
            if is_multiclass:
                metrics['roc_auc'].append(roc_auc_score(y_boot, y_prob, multi_class='ovr'))
            else:
                metrics['roc_auc'].append(roc_auc_score(y_boot, y_prob[:, 1]))

            metrics['f1'].append(f1_score(y_boot, y_pred, average=avg_method, zero_division=0))
            metrics['accuracy'].append(accuracy_score(y_boot, y_pred))
            metrics['precision'].append(precision_score(y_boot, y_pred, average=avg_method, zero_division=0))
            metrics['recall'].append(recall_score(y_boot, y_pred, average=avg_method, zero_division=0))
        except:
            continue

    results = {}
    for k, v in metrics.items():
        if len(v) == 0:
            results[k] = "N/A"
        else:
            mean_val = np.mean(v)
            ci_val = 1.96 * np.std(v)
            results[k] = f"{mean_val:.4f}±{ci_val:.4f}"

    return results

In [ ]:
def load_and_prep_data(dataset_name):
    # Маппинг ID датасетов в OpenML
    dataset_ids = {
        'Bank': 1461, 'Blood': 1464, 'California': 44090,
        'Car': 40975, 'Credit-G': 31, 'Diabetes': 37,
        'Heart': 53, 'Income': 1590, 'Jungle': 41027
    }

    if dataset_name == 'Diabetes':
        path = kagglehub.dataset_download("uciml/pima-indians-diabetes-database") + "/diabetes.csv"
        df = pd.read_csv(path)
        target_name = 'Outcome'
        X = df.drop(columns=[target_name])
        y = df[target_name]
    elif dataset_name == 'Heart':
        path = kagglehub.dataset_download("fedesoriano/heart-failure-prediction") + "/heart.csv"
        df = pd.read_csv(path)
        target_name = 'HeartDisease'
        X = df.drop(columns=[target_name])
        y = df[target_name]
    else:
        data = fetch_openml(data_id=dataset_ids[dataset_name], as_frame=True, parser='auto')
        X, y = data.data, data.target

    # Обработка пропущенных значений
    for col in X.columns:
        if X[col].dtype.name in ['category', 'object']:
            X[col] = X[col].astype(str).fillna('Missing')
        else:
            X[col] = X[col].fillna(X[col].median())

    # Label Encoding для целевой переменной
    le = LabelEncoder()
    y = le.fit_transform(y.astype(str))

    return X, y

In [ ]:
def run_standard_experiment(X, y, dataset_name):
    X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val)
    cat_features = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()

    model = CatBoostClassifier(iterations=1000, learning_rate=0.1, depth=6,
                               cat_features=cat_features, verbose=0)
    model.fit(X_train, y_train)

    results = evaluate_with_bootstrap(model, X_test, y_test)

    row = {'Dataset': dataset_name}
    row.update(results)

    return row

In [ ]:
def get_fast_preprocessor(X):
    cat_cols = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    num_cols = X.select_dtypes(exclude=['object', 'category', 'string']).columns.tolist()

    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ])
    return preprocessor

def run_logistic_regression(X_train, X_test, y_train, y_test):
    pipe = Pipeline([
        ('prep', get_fast_preprocessor(X_train)),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ])
    pipe.fit(X_train, y_train)
    return evaluate_with_bootstrap(pipe, X_test, y_test)

def run_lightgbm(X_train, X_test, y_train, y_test):
    # Создаем копии, чтобы не изменять исходные данные
    X_train_lgb = X_train.copy()
    X_test_lgb = X_test.copy()

    # LightGBM требует тип 'category' вместо 'object'/'string'
    cat_cols = X_train_lgb.select_dtypes(include=['object', 'string']).columns
    for col in cat_cols:
        X_train_lgb[col] = X_train_lgb[col].astype('category')
        X_test_lgb[col] = X_test_lgb[col].astype('category')

    model = LGBMClassifier(n_estimators=1000, random_state=42, verbose=-1, device='gpu')

    model.fit(X_train_lgb, y_train)
    return evaluate_with_bootstrap(model, X_test_lgb, y_test)

def run_catboost(X_train, X_test, y_train, y_test):
    cat_features = X_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    model = CatBoostClassifier(iterations=1000, learning_rate=0.1, depth=6, cat_features=cat_features,
                               verbose=0, random_seed=42, nan_mode='Min', task_type='GPU')
    model.fit(X_train, y_train)
    return evaluate_with_bootstrap(model, X_test, y_test)

In [ ]:
datasets = ["Bank", "Blood", "California", "Car", "Credit-G", "Diabetes", "Heart", "Income", "Jungle"]
Logistic_regression_results = []

for ds_name in tqdm(datasets, "Logistic_regression"):
    X, y = load_and_prep_data(ds_name)
    X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val)
    res_lr = run_logistic_regression(X_train, X_test, y_train, y_test)
    Logistic_regression_results.append({'Dataset': ds_name, **res_lr})

# Вывод итогового DataFrame с результатами
Logistic_regression_df = pd.DataFrame(Logistic_regression_results)
Logistic_regression_df

Logistic_regression:  56%|█████▌    | 5/9 [03:37<02:06, 31.67s/it]

Using Colab cache for faster access to the 'pima-indians-diabetes-database' dataset.


Logistic_regression:  67%|██████▋   | 6/9 [03:55<01:21, 27.07s/it]

Using Colab cache for faster access to the 'heart-failure-prediction' dataset.


Logistic_regression: 100%|██████████| 9/9 [06:45<00:00, 45.09s/it]


,Dataset,roc_auc,f1,accuracy,precision,recall
0,Bank,0.9054±0.0090,0.4503±0.0300,0.9013±0.0060,0.6479±0.0407,0.3452±0.0281
1,Blood,0.7846±0.0903,0.1794±0.1524,0.7671±0.0690,0.5659±0.4052,0.1092±0.1014
2,California,0.9137±0.0084,0.8374±0.0117,0.8363±0.0111,0.8321±0.0163,0.8428±0.0151
3,Car,0.9873±0.0060,0.7874±0.0831,0.9016±0.0306,0.8381±0.0782,0.7566±0.0943
4,Credit-G,0.7592±0.0714,0.8042±0.0499,0.7191±0.0616,0.7828±0.0641,0.8280±0.0635
5,Diabetes,0.8232±0.0654,0.5710±0.1203,0.7277±0.0707,0.6374±0.1470,0.5211±0.1346
6,Heart,0.9224±0.0436,0.8946±0.0447,0.8801±0.0485,0.8696±0.0642,0.9220±0.0530
7,Income,0.9053±0.0063,0.6600±0.0166,0.8533±0.0071,0.7408±0.0198,0.5952±0.0197
8,Jungle,0.8092±0.0072,0.5035±0.0111,0.6759±0.0101,0.5464±0.0233,0.5090±0.0086


In [ ]:
datasets = ["Bank", "Blood", "California", "Car", "Credit-G", "Diabetes", "Heart", "Income", "Jungle"]
CatBoost_results = []

for ds_name in tqdm(datasets, "CatBoost"):
    X, y = load_and_prep_data(ds_name)
    X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val)
    res_cat = run_catboost(X_train, X_test, y_train, y_test)
    CatBoost_results.append({'Dataset': ds_name, **res_cat})

# Вывод итогового DataFrame с результатами
CatBoost_df = pd.DataFrame(CatBoost_results)
CatBoost_df

CatBoost:  56%|█████▌    | 5/9 [04:56<03:12, 48.10s/it]

Using Colab cache for faster access to the 'pima-indians-diabetes-database' dataset.


CatBoost:  67%|██████▋   | 6/9 [05:38<02:17, 45.83s/it]

Using Colab cache for faster access to the 'heart-failure-prediction' dataset.


CatBoost: 100%|██████████| 9/9 [09:42<00:00, 64.69s/it]


,Dataset,roc_auc,f1,accuracy,precision,recall
0,Bank,0.9307±0.0066,0.5479±0.0261,0.9083±0.0058,0.6483±0.0333,0.4747±0.0286
1,Blood,0.7426±0.0960,0.4530±0.1500,0.7795±0.0659,0.5591±0.1911,0.3870±0.1578
2,California,0.9616±0.0053,0.8935±0.0099,0.8928±0.0096,0.8873±0.0133,0.8998±0.0134
3,Car,0.9985±0.0019,0.9471±0.0466,0.9798±0.0151,0.9664±0.0393,0.9339±0.0613
4,Credit-G,0.7597±0.0763,0.8403±0.0440,0.7558±0.0598,0.7728±0.0637,0.9217±0.0432
5,Diabetes,0.8006±0.0711,0.5969±0.1108,0.7282±0.0707,0.6240±0.1322,0.5759±0.1298
6,Heart,0.9347±0.0382,0.8834±0.0474,0.8742±0.0477,0.9074±0.0580,0.8615±0.0656
7,Income,0.9300±0.0053,0.7186±0.0158,0.8781±0.0067,0.8026±0.0184,0.6506±0.0194
8,Jungle,0.9717±0.0021,0.8086±0.0108,0.8584±0.0072,0.8138±0.0117,0.8040±0.0116


In [ ]:
datasets = ["Bank", "Blood", "California", "Car", "Credit-G", "Diabetes", "Heart", "Income", "Jungle"]
LightGBM_results = []

for ds_name in tqdm(datasets, "LightGBM"):
    X, y = load_and_prep_data(ds_name)
    X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val)
    res_lgb = run_lightgbm(X_train, X_test, y_train, y_test)
    LightGBM_results.append({'Dataset': ds_name, **res_lgb})

# Вывод итогового DataFrame с результатами
LightGBM_df = pd.DataFrame(LightGBM_results)
LightGBM_df

LightGBM:  56%|█████▌    | 5/9 [11:35<06:38, 99.57s/it] 

Using Colab cache for faster access to the 'pima-indians-diabetes-database' dataset.


LightGBM:  67%|██████▋   | 6/9 [11:55<03:37, 72.53s/it]

Using Colab cache for faster access to the 'heart-failure-prediction' dataset.


LightGBM: 100%|██████████| 9/9 [32:17<00:00, 215.28s/it]


,Dataset,roc_auc,f1,accuracy,precision,recall
0,Bank,0.9240±0.0070,0.5424±0.0275,0.9068±0.0059,0.6375±0.0327,0.4722±0.0306
1,Blood,0.6992±0.0953,0.4406±0.1460,0.7527±0.0673,0.4801±0.1743,0.4132±0.1588
2,California,0.9654±0.0049,0.9032±0.0096,0.9027±0.0093,0.8983±0.0129,0.9082±0.0126
3,Car,0.9990±0.0019,0.9556±0.0510,0.9885±0.0113,0.9730±0.0406,0.9424±0.0620
4,Credit-G,0.6863±0.0812,0.7946±0.0510,0.7059±0.0622,0.7771±0.0659,0.8139±0.0647
5,Diabetes,0.8033±0.0698,0.6313±0.1078,0.7527±0.0687,0.6581±0.1315,0.6108±0.1298
6,Heart,0.9269±0.0389,0.8799±0.0483,0.8701±0.0493,0.8993±0.0581,0.8624±0.0690
7,Income,0.9210±0.0057,0.6990±0.0161,0.8665±0.0069,0.7590±0.0191,0.6478±0.0201
8,Jungle,0.9749±0.0019,0.8012±0.0107,0.8571±0.0071,0.8057±0.0116,0.7970±0.0114


In [ ]:
from google.colab import runtime
runtime.unassign()

Время выполнения:

- logistic regression ~ 7 мин,
- catboost ~ 10 мин,
- lightGBM ~ 26 мин.

Графический процессор A100 75.3/80GB.